## Objetivo del ejercicio: 
Optimización de hiperparámetros multivariable: k, weights, metric del algoritmo KNN.

#### Parametros de KNN:

* n_neighbors : k
* weights : "uniform" o "distance"
* metric : "euclidean" o "manhattan"

In [16]:
#Definicion del tipo de problema (variable continua) y como se representa una solucion (continua tambien)
from jmetal.core.problem import FloatProblem
from jmetal.core.solution import FloatSolution

#Utilizar un algoritmo genetico monoobjetivo
from jmetal.algorithm.singleobjective.genetic_algorithm import GeneticAlgorithm

#Operadores geneticos: crossover y mutacion
from jmetal.operator.crossover import SBXCrossover
from jmetal.operator.mutation import PolynomialMutation

#Como detener el algoritmo: iteraciones
from jmetal.util.termination_criterion import StoppingByEvaluations

#sklearn: Modelo: KNN Evaluación; validación cruzada; Dataset: Iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_iris

#Definicion del problema: Encontrar el mejor valor de k para KNN
class KNNProblem(FloatProblem):

    def __init__(self):
        super().__init__()

        #Se debinen los limites. En el caso de las variables de weights y metric se utiliza cun label encode [0,1] para sus opciones
        self.lower_bound = [1, 0, 0]
        self.upper_bound = [20, 1, 1]

        self.data = load_iris()

    def number_of_variables(self):
        return 3

    def number_of_objectives(self):
        return 1

    def number_of_constraints(self):
        return 0

    def evaluate(self, solution):
        k = max(1, int(round(solution.variables[0])))

        weights = "uniform" if round(solution.variables[1]) == 0 else "distance"
        metric = "euclidean" if round(solution.variables[2]) == 0 else "manhattan"

        model = KNeighborsClassifier(
            n_neighbors=k,
            weights=weights,
            metric=metric
        )

        score = cross_val_score(
            model,
            self.data.data,
            self.data.target,
            cv=3
        ).mean()

        solution.objectives[0] = -score
        return solution

    def create_solution(self):
        return FloatSolution(
            self.lower_bound,
            self.upper_bound,
            self.number_of_objectives()
        )

    def name(self):
        return "KNN: Optimizacion de hiperparametros multivariable"

In [17]:


# ===== ejecutar =====
#Se debe respetar la secuencia definida en jMetal

#1) Crear un problema
problem = KNNProblem()

algorithm = GeneticAlgorithm(
    problem=problem,
    population_size=10,                                                     #10 soluciones por generación
    offspring_population_size=10,                                           #10 nuevas soluciones por iteración
    mutation=PolynomialMutation(probability=0.2, distribution_index=20),    #20% de probabilidad de alterar una solución (cambiar un 1 por un 2)
    crossover=SBXCrossover(probability=0.9, distribution_index=20),         #90% de probabilidad de mezclar soluciones
    termination_criterion=StoppingByEvaluations(max_evaluations=50)         #Detener después de evaluar 50 soluciones
)

#Se ejecuta el algoritmo genetico: Se realiza:
#1) Genera soluciones
#2) Evalúa (función evaluate)
#3) Selecciona mejores
#4) Cruza y muta
#5) Repite hasta terminar
algorithm.run()

#Obtiene la mejor solución encontrada
result = algorithm.result()

#Traducir el label encode de los parametros
best_weights = "uniform" if round(result.variables[1]) == 0 else "distance"
best_metric = "euclidean" if round(result.variables[2]) == 0 else "manhattan"

print("Mejor k:", int(result.variables[0]))
print("Mejor weights:", int(result.variables[1]) , best_weights)
print("Mejor metric:", int(result.variables[2]), best_metric)

print("Accuracy:", -result.objectives[0])
#En las pruebas realizadas: Mejor k: 3 Accuracy: 0.9866666666666667. Este valor cambia entre ejecuciones

[2026-04-28 23:43:58,442] [jmetal.core.algorithm] [DEBUG] Creating initial set of solutions...
[2026-04-28 23:43:58,443] [jmetal.core.algorithm] [DEBUG] Evaluating solutions...
[2026-04-28 23:43:58,515] [jmetal.core.algorithm] [DEBUG] Initializing progress...
[2026-04-28 23:43:58,516] [jmetal.core.algorithm] [DEBUG] Running main loop until termination criteria is met
[2026-04-28 23:43:58,782] [jmetal.core.algorithm] [DEBUG] Finished!


Mejor k: 0
Mejor weights: 0 uniform
Mejor metric: 0 euclidean
Accuracy: 0.96
